<a href="https://colab.research.google.com/github/MiguelAngeloTr/AppCine/blob/main/C3/ParcialFinal/MiguelAngelJimenezTrochez_Parcial3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Examen Final
# Miguel Angel Jimenez Trochez- 2215407

**Duración estimada**: 3 horas  
**Dataset**: `Flight_delay.csv` (cargar mediante Kaggle)  
**Objetivo**: Evaluar habilidades en análisis exploratorio de datos, limpieza, y modelado en grandes bases de datos con PySpark.


Utilice el siguiente dataset para dar respuesta a cada sección. El dataset contiene datos sobre despegues y retrazo en vuelos en los aeropuertos.

In [1]:
# Configuración del entorno
!pip install -q pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("BigDataFinalExam").getOrCreate()

In [ ]:
!pip install -q kaggle

from google.colab import files
files.upload()  # Subir archivo kaggle.json con las credenciales

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d undersc0re/flight-delay-and-causes


In [ ]:
# Descomprimir
!unzip -o flight-delay-and-causes.zip

In [ ]:
!ls

In [ ]:
# Carga de datos
df = spark.read.csv("/content/Flight_delay.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()

## Sección 1 – Análisis Exploratorio de Datos (EDA) – **1.25 punto**

**1. ¿Cuántas filas y columnas tiene el dataset?**

R//Filas: 484 551, Columnas: 29

**2. Muestre los tipos de datos de cada columna.**

R//

**3. ¿Cuáles son las 5 aerolíneas (`Airline`) con mayor número de vuelos?**

R//

1.   Southwest Airlines Co.: 119 04
2.   American Airlines Inc.: 73 053
3.   American Eagle Airlines: 58 698
4.   United Air Lines Inc.: 56 896
5.   SkyWest Airlines Inc.: 50 384



**4. ¿Cuál es el promedio de retraso de llegada (`ArrDelay`) por aerolínea?**

R//
1. JetBlue Airways : 72.87 min
2. United Air Lines Inc. : 69.67 min
3. American Airlines Inc. : 65.73 min
4. SkyWest Airlines Inc. : 65.19 min
5. American Eagle Airlines : 64.28 min
(y sigue con Atlantic Southeast, Delta, US Airways, Alaska, Hawaiian…)

**5. ¿Cuáles son los 5 aeropuertos de origen (`Origin`) con mayor retraso promedio en la salida (`DepDelay`)?**

R//
1. ACY: 195.80 min
2. EGE: 103.57 min
3. SLE: 103.29 min
4. SPI: 100.25 min
5. ABI: 96.82 min

**6. ¿Cuál es la proporción de vuelos cancelados (Cancelled) y desviados (Diverted)?**

R//
Solo el 0 % de los vuelos aparecen como cancelados o desviados.


In [ ]:
from pyspark.sql import functions as F


1. Número de filas y columnas

In [ ]:
# Count de filas
num_rows = df.count()

# Count de columnas
num_cols = len(df.columns)

print(f"Filas: {num_rows}, Columnas: {num_cols}")


Filas: 484551, Columnas: 29


2. Tipos de datos de cada columna

In [ ]:
df.printSchema()

root
 |-- DayOfWeek: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- Airline: string (nullable = true)
 |-- FlightNum: integer (nullable = true)
 |-- TailNum: string (nullable = true)
 |-- ActualElapsedTime: integer (nullable = true)
 |-- CRSElapsedTime: integer (nullable = true)
 |-- AirTime: integer (nullable = true)
 |-- ArrDelay: integer (nullable = true)
 |-- DepDelay: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Org_Airport: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Dest_Airport: string (nullable = true)
 |-- Distance: integer (nullable = true)
 |-- TaxiIn: integer (nullable = true)
 |-- TaxiOut: integer (nullable = true)
 |-- Cancelled: integer (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: integer (nullable = true

3. Top 5 aerolíneas por número de vuelos

In [ ]:
top5_airlines = (
    df.groupBy("Airline")
      .count()
      .orderBy(F.desc("count"))
      .limit(5)
)
top5_airlines.show()


+--------------------+------+
|             Airline| count|
+--------------------+------+
|Southwest Airline...|119048|
|American Airlines...| 73053|
|American Eagle Ai...| 58698|
|United Air Lines ...| 56896|
|Skywest Airlines ...| 50384|
+--------------------+------+



4. Promedio de retraso de llegada (ArrDelay) por aerolínea

In [ ]:
avg_arrdelay = (
    df.filter(F.col("ArrDelay").isNotNull())
      .groupBy("Airline")
      .agg(F.round(F.avg("ArrDelay"), 2).alias("avg_ArrDelay"))
      .orderBy(F.desc("avg_ArrDelay"))
)
avg_arrdelay.show()


+--------------------+------------+
|             Airline|avg_ArrDelay|
+--------------------+------------+
|     JetBlue Airways|       72.87|
|United Air Lines ...|       69.67|
|American Airlines...|       65.73|
|Skywest Airlines ...|       65.19|
|American Eagle Ai...|       64.28|
|Atlantic Southeas...|       63.21|
|Delta Air Lines Inc.|       59.29|
|     US Airways Inc.|       58.45|
|Alaska Airlines Inc.|       57.56|
|Hawaiian Airlines...|       55.66|
|Southwest Airline...|       51.03|
|Frontier Airlines...|       41.97|
+--------------------+------------+



5. Top 5 aeropuertos de origen con mayor retraso promedio de salida (DepDelay)

In [ ]:
top5_origins = (
    df.filter(F.col("DepDelay").isNotNull())
      .groupBy("Origin")
      .agg(F.round(F.avg("DepDelay"), 2).alias("avg_DepDelay"))
      .orderBy(F.desc("avg_DepDelay"))
      .limit(5)
)
top5_origins.show()


+------+------------+
|Origin|avg_DepDelay|
+------+------------+
|   ACY|       195.8|
|   EGE|      103.57|
|   SLE|      103.29|
|   SPI|      100.25|
|   ABI|       96.82|
+------+------------+



6. Proporción de vuelos cancelados y desviados

In [ ]:
# Total de vuelos
total = df.count()

# Agrupar y contar
cancel_div = (
    df.groupBy("Cancelled", "Diverted")
      .count()
      .withColumn("pct",
          F.round(100 * F.col("count") / total, 2)
      )
)
cancel_div.show()


+---------+--------+------+-----+
|Cancelled|Diverted| count|  pct|
+---------+--------+------+-----+
|        0|       0|484551|100.0|
+---------+--------+------+-----+



## Sección 2 – Limpieza y Preparación de Datos – **0.75 puntos**

1. Elimine las filas con valores nulos en las columnas: `ArrDelay`, `DepDelay`, `Airline`, `Origin`, `Dest`.
2. Cree una nueva variable binaria llamada `IS_DELAYED`:
   - Valor 1 si `ArrDelay > 15` minutos
   - Valor 0 en caso contrario
3. Cree una columna `FlightHour` extrayendo la hora de salida programada (`DepTime`). Puede truncarla a la hora redondeada.

1. Eliminar filas con nulos en columnas críticas

In [ ]:
df_clean = df.dropna(subset=["ArrDelay", "DepDelay", "Airline", "Origin", "Dest"])

2. Crear variable binaria IS_DELAYED

In [ ]:
df_clean = df_clean.withColumn(
    "IS_DELAYED",
    F.when(F.col("ArrDelay") > 15, F.lit(1)).otherwise(F.lit(0))
)
# Mostrar un conteo rápido de vuelos tardíos vs a tiempo
df_clean.groupBy("IS_DELAYED").count().show()


+----------+------+
|IS_DELAYED| count|
+----------+------+
|         1|471530|
|         0| 13021|
+----------+------+



3.Cree una columna FlightHour extrayendo la hora de salida programada (DepTime).

In [ ]:
df_clean = df_clean.withColumn("FlightHour", F.floor(F.col("DepTime") / 100))

# (Opcionalmente, versión más robusta con strings)
df_clean = (
    df_clean
    .withColumn("DepTimeStr", F.lpad(F.col("DepTime").cast("string"), 4, "0"))
    .withColumn("FlightHour",
        F.col("DepTimeStr").substr(1, 2).cast("integer")
    )
    .drop("DepTimeStr")
)

# Verificar los primeros registros
df_clean.select("DepTime", "FlightHour", "ArrDelay", "IS_DELAYED").show(5)

# 4. (Opcional) Esquema final
print("Esquema final de df_clean:")
df_clean.printSchema()


+-------+----------+--------+----------+
|DepTime|FlightHour|ArrDelay|IS_DELAYED|
+-------+----------+--------+----------+
|   1829|        18|      34|         1|
|   1937|        19|      57|         1|
|   1644|        16|      80|         1|
|   1452|        14|      15|         0|
|   1323|        13|      16|         1|
+-------+----------+--------+----------+
only showing top 5 rows

Esquema final de df_clean:
root
 |-- DayOfWeek: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- DepTime: integer (nullable = true)
 |-- ArrTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- Airline: string (nullable = true)
 |-- FlightNum: integer (nullable = true)
 |-- TailNum: string (nullable = true)
 |-- ActualElapsedTime: integer (nullable = true)
 |-- CRSElapsedTime: integer (nullable = true)
 |-- AirTime: integer (nullable = true)
 |-- ArrDelay: integer (nullable = true)
 |-- DepDelay: integer (nullab

## Sección 3 – Regresión (ML supervisado) – **1 punto**

Objetivo: Predecir el retraso en la llegada (ArrDelay) usando variables relevantes.

1. Entrene un modelo de regresión lineal para predecir `ArrDelay` considerando al menos las variables:
   - `DepDelay`, `FlightHour`, `Airline`, `Distance`, `AirTime`, `Origin`, `Dest`.
**2. Divida el dataset en entrenamiento (70%) y prueba (30%).**

R// Realicé train_df, test_df = df_clean.randomSplit([0.7, 0.3], seed=42) para obtener la partición 70/30 con semilla fija.

**3. Reporte el **RMSE** del modelo en el conjunto de prueba.**

R//
Tras entrenar y predecir sobre test_df, el RMSE en test fue 15.28 minutos.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

1. Partir en train (70%) / test (30%)

In [ ]:
train_df, test_df = df_clean.randomSplit([0.7, 0.3], seed=42)
print(f"▶️ Train rows: {train_df.count()}, Test rows: {test_df.count()}")

▶️ Train rows: 339092, Test rows: 145459


2. Índices y codificación de categorías

In [ ]:
categorical_cols = ["Airline", "Origin", "Dest"]
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_Idx", handleInvalid="keep")
    for col in categorical_cols
]
encoders = [
    OneHotEncoder(inputCol=f"{col}_Idx", outputCol=f"{col}_Vec")
    for col in categorical_cols
]

3. Vector de features

In [ ]:
feature_cols = ["DepDelay", "FlightHour", "Distance", "AirTime"] + [f"{c}_Vec" for c in categorical_cols]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

4. Modelo de regresión lineal

In [ ]:
lr = LinearRegression(labelCol="ArrDelay", featuresCol="features")

5. Pipeline

In [ ]:
pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

6. Entrenamiento

In [ ]:
print("⚙️  Entrenando modelo de regresión lineal…")
model = pipeline.fit(train_df)

⚙️  Entrenando modelo de regresión lineal…


7. Predicción

In [ ]:
predictions = model.transform(test_df)
predictions.select("ArrDelay", "prediction").show(5)

+--------+------------------+
|ArrDelay|        prediction|
+--------+------------------+
|      31|17.542293438115387|
|      40|44.804061172049074|
|      31|24.044868651297715|
|      43| 48.51681734031375|
|     255| 256.2176687137048|
+--------+------------------+
only showing top 5 rows



8. Evaluación con RMSE

In [ ]:
evaluator = RegressionEvaluator(
    labelCol="ArrDelay", predictionCol="prediction", metricName="rmse"
)
rmse = evaluator.evaluate(predictions)
print(f"✅ RMSE en test: {rmse:.2f}")

✅ RMSE en test: 15.28


In [ ]:
from pyspark.sql import functions as F
df_clean.select(F.round(F.stddev("ArrDelay"), 2).alias("stddev_ArrDelay")).show()


+---------------+
|stddev_ArrDelay|
+---------------+
|          56.98|
+---------------+



 El RMSE de 15.28 min equivale a solo un 26.8 % de la desviación estándar de 56.98 min. En otras palabras, tu modelo reduce el error en aproximadamente un 73.2 % frente a la variabilidad natural de los retrasos.

## Sección 4 – Clasificación (ML supervisado) – **1 punto**

Objetivo: Clasificar si un vuelo llegará tarde (`IsDelayed = 1`) o no.

**1. Use la columna `IS_DELAYED` como variable objetivo.**

R//
Utilicé IS_DELAYED (1 = retrasado, 0 = a tiempo) como etiqueta para el modelo.

**2. Entrene un modelo de clasificación considerando como predictores al menos las variables:
   - `DepDelay`, `DISTANCE`, `FlightHour`, `Airline`, `Origin`, `Dest`.**

R//
 Armé un pipeline con StringIndexer + OneHotEncoder para las categóricas (Airline, Origin, Dest), un VectorAssembler que incluyó DepDelay, Distance, FlightHour y los vectores codificados, y un LogisticRegression para clasificar IS_DELAYED.

**3. Divida el dataset en entrenamiento (70%) y prueba (30%)**
R//
Ejecuté train_df, test_df = df_clean.randomSplit([0.7, 0.3], seed=42) para garantizar reproducibilidad.

**4. Evalúe el modelo usando **accuracy**, **precision**, **recall** y **F1-score** y comente el resultado.**

R//

* Accuracy: 0.973
* Precision: 0.947
* Recall: 0.973
* F1-score: 0.960

Estas métricas muestran que el clasificador acierta el 97.3 % de los vuelos, identifica correctamente el 94.7 % de los retrasados (baja tasa de falsos positivos) y detecta el 97.3 % de los retrasos reales (mínimos falsos negativos). Un F1-score de 96.0 % confirma un excelente equilibrio entre precision y recall, lo que indica un modelo muy fiable.

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

 1. Split train/test

In [ ]:
train_df, test_df = df_clean.randomSplit([0.7, 0.3], seed=42)


2. Index y OneHotEncode de categóricas

In [ ]:
categorical_cols = ["Airline", "Origin", "Dest"]
indexers = [ StringIndexer(inputCol=c, outputCol=f"{c}_Idx", handleInvalid="keep")
             for c in categorical_cols ]
encoders = [ OneHotEncoder(inputCol=f"{c}_Idx", outputCol=f"{c}_Vec")
             for c in categorical_cols ]

3. Armar vector de features

In [ ]:
feature_cols = ["DepDelay", "Distance", "FlightHour"] + [f"{c}_Vec" for c in categorical_cols]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

4. Clasificador

In [ ]:
lr = LogisticRegression(labelCol="IS_DELAYED", featuresCol="features", maxIter=10)

5. Pipeline

In [ ]:
pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

6. Entrenar

In [ ]:
model = pipeline.fit(train_df)


7. Predecir

In [ ]:
preds = model.transform(test_df)

8. Evaluación

In [ ]:
evaluator_acc   = MulticlassClassificationEvaluator(labelCol="IS_DELAYED", predictionCol="prediction", metricName="accuracy")
evaluator_prec  = MulticlassClassificationEvaluator(labelCol="IS_DELAYED", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec   = MulticlassClassificationEvaluator(labelCol="IS_DELAYED", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1    = MulticlassClassificationEvaluator(labelCol="IS_DELAYED", predictionCol="prediction", metricName="f1")

accuracy  = evaluator_acc.evaluate(preds)
precision = evaluator_prec.evaluate(preds)
recall    = evaluator_rec.evaluate(preds)
f1_score  = evaluator_f1.evaluate(preds)

print(f" Accuracy : {accuracy:.3f}")
print(f" Precision: {precision:.3f}")
print(f" Recall   : {recall:.3f}")
print(f" F1-score : {f1_score:.3f}")

 Accuracy : 0.973
 Precision: 0.947
 Recall   : 0.973
 F1-score : 0.960


El clasificador es muy fiable: con un 97.3 % de accuracy acierta casi todos los casos, y su precision (94.7 %) y recall (97.3 %) demuestran que identifica correctamente vuelos retrasados con muy pocos falsos positivos y negativos. Un F1-score de 96.0 % confirma el excelente balance entre precisión y cobertura, lo que lo hace apto para producción siempre que se valide en datos nuevos.

## Sección 5 – Clustering (ML no supervisado) – **1 punto**

Objetivo: Agrupar vuelos con patrones similares de operación o retraso.

**1. Seleccione las variables `ArrDelay` y `DepDelay`. Si lo considera, puede usar otra variable numerica.**

R//Seleccioné ArrDelay y DepDelay como variables principales.

**2. Estandarice las variables y aplique un modelo de clustering sobre una muestra de vuelos.**

R//Estandaricé ambas con StandardScaler (media 0, desviación 1) y apliqué K-Means sobre una muestra de 30 000 registros.

**3. ¿Cuántos clusters parecen óptimos? Justifique su respuesta.**

R//Probé k=2…6 y el coeficiente de silhouette fue máximo (0.8361) en k=2, lo que indica la mejor cohesión y separación de grupos.

**4. Reporte el número de elementos por clúster.**

R//
Cluster 0: 4 225 vuelos
Cluster 1: 25 812 vuelos

**5. Describa brevemente las características de cada cluster.**

R//
Cluster 0 (crítico): vuelos con retrasos muy elevados (promedio DepDelay ≈ 164 min, ArrDelay ≈ 170 min).

Cluster 1 (moderado): vuelos con demoras intermedias (DepDelay ≈ 40 min, ArrDelay ≈ 43 min).



In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

1. Tomar muestra (p. ej. 30 000 vuelos)

In [ ]:
sample_df = df_clean.select("ArrDelay", "DepDelay").sample(False, 30000/df_clean.count(), seed=42)

2. Armar vector

In [ ]:
assembler = VectorAssembler(inputCols=["ArrDelay", "DepDelay"], outputCol="rawFeatures")
sample_feats = assembler.transform(sample_df)

3. Estandarizar

In [ ]:
scaler = StandardScaler(inputCol="rawFeatures", outputCol="features", withMean=True, withStd=True)
scaler_model = scaler.fit(sample_feats)
sample_scaled = scaler_model.transform(sample_feats)

4. Evaluar silhouette para k=2..6

In [ ]:
evaluator = ClusteringEvaluator(metricName="silhouette", featuresCol="features")
scores = []
for k in range(2, 7):
    km = KMeans(k=k, seed=42, featuresCol="features")
    model_k = km.fit(sample_scaled)
    preds_k = model_k.transform(sample_scaled)
    sil = evaluator.evaluate(preds_k)
    scores.append((k, sil))
    print(f"-> k={k}, silhouette={sil:.4f}")

# Suponiendo k_opt es el k con mejor silhouette
k_opt, _ = max(scores, key=lambda x: x[1])
print(f"\n Número óptimo de clusters según silhouette: {k_opt}")

-> k=2, silhouette=0.8361
-> k=3, silhouette=0.7882
-> k=4, silhouette=0.7494
-> k=5, silhouette=0.7335
-> k=6, silhouette=0.7165

 Número óptimo de clusters según silhouette: 2


5. Ajuste final y conteo

In [ ]:
final_km = KMeans(k=k_opt, seed=42, featuresCol="features")
final_model = final_km.fit(sample_scaled)
final_preds = final_model.transform(sample_scaled)

counts = final_preds.groupBy("prediction").count().orderBy("prediction")
counts.show()

+----------+-----+
|prediction|count|
+----------+-----+
|         0| 4225|
|         1|25812|
+----------+-----+



6. Características promedio por cluster

In [ ]:
final_preds.groupBy("prediction") \
    .agg(
       F.round(F.avg("ArrDelay"), 2).alias("avg_ArrDelay"),
       F.round(F.avg("DepDelay"), 2).alias("avg_DepDelay")
    ) \
    .orderBy("prediction") \
    .show()

+----------+------------+------------+
|prediction|avg_ArrDelay|avg_DepDelay|
+----------+------------+------------+
|         0|      169.94|      164.21|
|         1|       43.24|       40.34|
+----------+------------+------------+



Se agruparon los vuelos usando únicamente los retrasos de salida y llegada (DepDelay, ArrDelay), estandarizamos ambas variables y aplicamos K-Means probando de 2 a 6 clusters, evaluando el coeficiente de silhouette, que alcanzó su valor máximo (0.8361) en k = 2, por lo que elegimos dos grupos. El cluster 0 agrupa 4 225 vuelos con demoras muy severas (retraso promedio de salida ≈ 164 min y de llegada ≈ 170 min), mientras que el cluster 1, con 25 812 vuelos, incluye aquellos con retrasos moderados (≈ 40 min en salida y ≈ 43 min en llegada). Esta partición permite distinguir claramente entre vuelos críticamente retrasados y la mayoría que sufre demoras moderadas, facilitando focalizar intervenciones donde más se necesitan.